# THỰC HÀNH CHƯƠNG 3: PHÂN ĐOẠN ẢNH (IMAGE SEGMENTATION)
**Môn học:** Thị giác máy tính (INFO3111)

## PHẦN 1: KHỞI TẠO MÔI TRƯỜNG VÀ TIỀN XỬ LÝ ẢNH

### 1.1. Mục tiêu và Lý thuyết
Phân đoạn ảnh là quá trình chia một bức ảnh thành nhiều vùng có ý nghĩa bằng cách gán nhãn cho các pixel (dựa trên màu sắc, cường độ sáng, v.v.). Đây là bước cực kỳ quan trọng cho các bài toán bậc cao như nhận dạng đối tượng hay xe tự lái.

**Quy trình chuẩn bị phân đoạn:**
Trước khi áp dụng bất kỳ thuật toán phân đoạn nào (như Cắt ngưỡng hay K-Means), chúng ta cần thực hiện **Tiền xử lý (Pre-processing)** để làm sạch dữ liệu:
1. **Chuyển đổi ảnh xám (Grayscale):** Giảm độ phức tạp tính toán từ 3 kênh màu (RGB) xuống 1 kênh cường độ sáng, rất cần thiết cho các thuật toán dựa trên biểu đồ Histogram.
2. **Lọc nhiễu (Blur/Smoothing):** Sử dụng bộ lọc (ví dụ: Gaussian Blur) để làm mờ các chi tiết nhiễu li ti, giúp các vùng phân đoạn trở nên liền mạch hơn và tránh bị "vụn".

In [ ]:
# Cài đặt (nếu chạy trên máy cá nhân) và Import các thư viện cốt lõi
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Hàm hỗ trợ hiển thị ảnh trên Jupyter/Colab
def show_images(images, titles, rows=1, cols=2, figsize=(12, 5)):
    """
    Hàm hiển thị nhiều ảnh cùng lúc sử dụng Matplotlib.
    Tự động xử lý chuyển đổi không gian màu từ BGR (OpenCV) sang RGB (Matplotlib) hoặc ảnh xám.
    """
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten() if isinstance(axes, np.ndarray) else [axes]

    for i, (img, title) in enumerate(zip(images, titles)):
        if len(img.shape) == 3: # Nếu là ảnh màu (3 kênh)
            # Chuyển BGR (mặc định của cv2) sang RGB để matplotlib hiển thị đúng màu
            img_disp = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[i].imshow(img_disp)
        else: # Nếu là ảnh xám (1 kênh)
            axes[i].imshow(img, cmap='gray')

        axes[i].set_title(title)
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

print("✅ Đã import thư viện và khởi tạo hàm hiển thị thành công!")

### 1.2. Tải ảnh và Thực hiện Tiền xử lý
Trong phần này, chúng ta sẽ tải một bức ảnh lên, sau đó thực hiện hai bước tiền xử lý cơ bản:
- `cv2.cvtColor()`: Để chuyển ảnh sang thang độ xám.
- `cv2.GaussianBlur()`: Để làm mượt ảnh. Hàm này yêu cầu tham số kích thước kernel (phải là số lẻ, ví dụ: `(5, 5)`) và độ lệch chuẩn (thường để `0` để tự động tính toán).

*Lưu ý: Nếu sử dụng Google Colab, hãy tải ảnh của bạn (ví dụ: `image_sample.jpg`) lên thư mục bên trái trước khi chạy đoạn code dưới.*

In [ ]:
# 1. Đọc ảnh từ hệ thống
# Thay 'image_sample.jpg' bằng tên file ảnh bạn đã tải lên (vd: ảnh tài liệu, ảnh chiếc bát)
image_path = 'image_sample.jpg'

# Tạo một ảnh giả lập (ảnh nhiễu) trong trường hợp chưa kịp tải ảnh lên để code không bị lỗi
# (Bạn có thể comment đoạn tạo ảnh giả này lại khi đã có ảnh thật)
dummy_img = np.random.randint(0, 255, (300, 300, 3), dtype=np.uint8)
cv2.imwrite('image_sample.jpg', dummy_img)

# Đọc ảnh gốc bằng OpenCV
img_original = cv2.imread(image_path)

if img_original is None:
    print("❌ Lỗi: Không tìm thấy ảnh. Vui lòng kiểm tra lại đường dẫn/tên file!")
else:
    # 2. Chuyển đổi sang ảnh xám
    img_gray = cv2.cvtColor(img_original, cv2.COLOR_BGR2GRAY)

    # 3. Làm mượt ảnh bằng Gaussian Blur để giảm nhiễu trước khi phân đoạn
    # Kernel size (5,5) là kích thước ma trận lọc, số càng lớn ảnh càng mờ
    img_blurred = cv2.GaussianBlur(img_gray, (5, 5), 0)

    # 4. Hiển thị so sánh
    images = [img_original, img_gray, img_blurred]
    titles = ['Ảnh Gốc (RGB)', 'Ảnh Xám (Grayscale)', 'Ảnh Xám Đã Làm Mượt (Gaussian Blur)']

    show_images(images, titles, rows=1, cols=3, figsize=(15, 5))

## PHẦN 2: PHÂN ĐOẠN CƠ BẢN VỚI CẮT NGƯỠNG TOÀN CỤC VÀ THUẬT TOÁN OTSU

### 2.1. Cắt ngưỡng Toàn cục (Global Thresholding)
Cắt ngưỡng là phương pháp đơn giản nhất để tách vật thể ra khỏi nền. Ý tưởng là chọn một giá trị ngưỡng $T$ (từ 0 đến 255). Tất cả các pixel sáng hơn $T$ sẽ được chuyển thành màu trắng (255), và ngược lại sẽ thành màu đen (0).

**Công thức toán học:**
$$g(x,y) = \begin{cases} 255 & \text{nếu } f(x,y) > T \\ 0 & \text{nếu } f(x,y) \le T \end{cases}$$

Để chọn được $T$ thủ công, chúng ta thường nhìn vào **Biểu đồ Histogram** (biểu đồ thể hiện tần suất xuất hiện của các mức sáng trong ảnh). Nếu ảnh có độ tương phản tốt, biểu đồ sẽ có 2 đỉnh rõ rệt (Bimodal), và $T$ nằm ở "thung lũng" giữa 2 đỉnh đó.

In [ ]:
# 1. Vẽ biểu đồ Histogram của ảnh xám đã làm mượt
plt.figure(figsize=(10, 4))
plt.hist(img_blurred.ravel(), bins=256, range=[0, 256], color='gray')
plt.title('Biểu đồ Histogram của ảnh')
plt.xlabel('Cường độ sáng (0 - 255)')
plt.ylabel('Số lượng Pixel')
plt.axvline(x=127, color='r', linestyle='--', label='Ngưỡng T=127 (Thử nghiệm)')
plt.legend()
plt.show()

# 2. Cắt ngưỡng thủ công (Thử với T = 127)
# Hàm cv2.threshold trả về 2 giá trị: ret (giá trị ngưỡng đã dùng) và thresh_img (ảnh kết quả)
T_manual = 127
ret, thresh_manual = cv2.threshold(img_blurred, T_manual, 255, cv2.THRESH_BINARY)

# 3. Thử nghiệm với một ngưỡng T khác (ví dụ T = 200) để thấy sự khác biệt
T_high = 200
_, thresh_high = cv2.threshold(img_blurred, T_high, 255, cv2.THRESH_BINARY)

# Hiển thị so sánh
show_images(
    [img_gray, thresh_manual, thresh_high],
    ['Ảnh Xám Gốc', f'Cắt ngưỡng (T={T_manual})', f'Cắt ngưỡng (T={T_high})'],
    cols=3
)

### 2.2. Thuật toán Otsu (Tự động tìm ngưỡng tối ưu)
**Vấn đề:** Việc đoán mò giá trị $T$ như trên (127 hay 200) rất mất thời gian và không thể áp dụng tự động cho hàng ngàn bức ảnh khác nhau.

**Giải pháp:** Thuật toán **Otsu** sẽ tự động phân tích biểu đồ Histogram và tính toán ra một giá trị $T$ tối ưu nhất sao cho phương sai giữa hai lớp (nền và vật thể) là lớn nhất (tức là tách biệt rõ ràng nhất).

Trong OpenCV, chúng ta chỉ cần cộng thêm cờ `cv2.THRESH_OTSU` vào tham số loại cắt ngưỡng, và truyền giá trị ngưỡng ban đầu là `0` (vì máy sẽ tự tính lại).

In [ ]:
# Áp dụng thuật toán Otsu
# Truyền 0 vào vị trí của T, OpenCV sẽ bỏ qua số 0 này và tự tính T tối ưu nhờ cờ THRESH_OTSU
ret_otsu, thresh_otsu = cv2.threshold(img_blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

print(f"✅ Thuật toán Otsu đã tự động tìm ra ngưỡng tối ưu là: T = {ret_otsu}")

# Trực quan hóa kết quả của Otsu trên Histogram
plt.figure(figsize=(10, 4))
plt.hist(img_blurred.ravel(), bins=256, range=[0, 256], color='gray')
plt.title('Biểu đồ Histogram với Ngưỡng Otsu')
plt.axvline(x=ret_otsu, color='g', linestyle='-', linewidth=2, label=f'Ngưỡng Otsu T={ret_otsu}')
plt.legend()
plt.show()

# Hiển thị so sánh ảnh cắt thủ công và ảnh cắt bằng Otsu
show_images(
    [thresh_manual, thresh_otsu],
    [f'Cắt thủ công (T={T_manual})', f'Cắt bằng Otsu (T={ret_otsu})'],
    cols=2
)

## PHẦN 3: KHẮC PHỤC NHIỄU SÁNG VỚI CẮT NGƯỠNG THÍCH ỨNG (ADAPTIVE THRESHOLDING)

### 3.1. Vấn đề của Cắt ngưỡng Toàn cục
Trong thực tế, khi chụp ảnh tài liệu bằng điện thoại, chúng ta rất hay gặp tình trạng **chiếu sáng không đồng đều** (chỗ sáng, chỗ bị đổ bóng).
Nếu sử dụng một ngưỡng $T$ duy nhất (Toàn cục hoặc Otsu) cho toàn bộ bức ảnh, thuật toán sẽ thất bại: nó có thể làm mất chữ ở vùng sáng hoặc làm đen kịt vùng bị đổ bóng.

Để minh họa, chúng ta sẽ tải một bức ảnh tài liệu bị bóng mờ và thử nghiệm lại với thuật toán Otsu.

In [ ]:
# 1. Tạo một ảnh tài liệu giả lập bị đổ bóng (nếu bạn chưa có ảnh thực tế)
# (Hãy comment đoạn này và đổi tên file ở dưới nếu bạn đã tải ảnh tài liệu lên Colab)
img_h, img_w = 300, 600
dummy_doc = np.ones((img_h, img_w), dtype=np.uint8) * 255
cv2.putText(dummy_doc, "DONG A UNIVERSITY", (30, 100), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 0), 4)
cv2.putText(dummy_doc, "COMPUTER VISION", (30, 200), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 0), 4)

# Thêm hiệu ứng đổ bóng (tối dần từ phải sang trái)
gradient = np.tile(np.linspace(0.3, 1.0, img_w), (img_h, 1))
dummy_doc_shadow = (dummy_doc * gradient).astype(np.uint8)
cv2.imwrite('document_sample.jpg', dummy_doc_shadow)

# ==========================================

# 2. Đọc ảnh tài liệu và tiền xử lý
doc_path = 'document_sample.jpg'
img_doc = cv2.imread(doc_path, cv2.IMREAD_GRAYSCALE)
img_doc_blurred = cv2.GaussianBlur(img_doc, (5, 5), 0)

# 3. Thử nghiệm thuật toán Otsu để xem sự thất bại
ret_otsu_doc, thresh_otsu_doc = cv2.threshold(img_doc_blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Hiển thị
show_images(
    [img_doc, thresh_otsu_doc],
    ['Ảnh tài liệu bị đổ bóng', f'Thất bại với Otsu (T={ret_otsu_doc})'],
    cols=2
)

### 3.2. Cắt ngưỡng Thích ứng (Adaptive Thresholding)
**Giải pháp:** Thay vì dùng 1 ngưỡng chung, ta sẽ cho máy tính trượt một "cửa sổ" (block) qua từng vùng nhỏ của ảnh. Ngưỡng sẽ được tính riêng cho từng vùng đó dựa trên các pixel xung quanh.

Hàm `cv2.adaptiveThreshold` có 2 phương pháp tính ngưỡng vùng lân cận:
* `cv2.ADAPTIVE_THRESH_MEAN_C`: Tính trung bình cộng.
* `cv2.ADAPTIVE_THRESH_GAUSSIAN_C`: Tính trung bình có trọng số (Gaussian), thường cho kết quả tự nhiên và ít nhiễu hơn.

**Các tham số quan trọng:**
* `blockSize`: Kích thước vùng lân cận (phải là số lẻ: 11, 15, 21...).
* `C`: Hằng số trừ đi từ giá trị trung bình vừa tính (giúp tinh chỉnh lại độ dày của chữ).

In [ ]:
# 1. Cắt ngưỡng thích ứng - Phương pháp Mean
thresh_mean = cv2.adaptiveThreshold(
    img_doc_blurred, 255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY,
    blockSize=21, C=10
)

# 2. Cắt ngưỡng thích ứng - Phương pháp Gaussian
thresh_gaussian = cv2.adaptiveThreshold(
    img_doc_blurred, 255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    blockSize=21, C=10
)

# Hiển thị so sánh
show_images(
    [thresh_otsu_doc, thresh_mean, thresh_gaussian],
    ['Otsu (Tham chiếu)', 'Adaptive - Mean', 'Adaptive - Gaussian'],
    cols=3
)

### 3.3. Chiến thuật Nâng cao: CLAHE + Cắt ngưỡng (Nhiệm vụ Lab 3)
Đôi khi, ánh sáng quá phức tạp khiến cả Adaptive Thresholding cũng để lại nhiễu hạt. Để đạt điểm tối đa trong Lab 3, chúng ta sử dụng một chiến thuật tiền xử lý mạnh mẽ hơn: **CLAHE (Contrast Limited Adaptive Histogram Equalization)**.

* **CLAHE làm gì?** Nó chia ảnh thành các ô nhỏ (grid) và tự động cân bằng lại độ tương phản cho từng ô đó, giúp làm đều ánh sáng toàn cục trước khi cắt ngưỡng.
* **Quy trình chuẩn:** Ảnh xám $\rightarrow$ CLAHE $\rightarrow$ Làm mượt (Blur) $\rightarrow$ Otsu / Adaptive.

In [ ]:
# 1. Khởi tạo đối tượng CLAHE
# clipLimit: Giới hạn độ tương phản (để không bị nhiễu quá mức). tileGridSize: Kích thước ô chia.
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

# 2. Áp dụng CLAHE lên ảnh xám gốc
img_clahe = clahe.apply(img_doc)

# 3. Làm mượt ảnh sau khi cân bằng sáng
img_clahe_blurred = cv2.GaussianBlur(img_clahe, (5, 5), 0)

# 4. Áp dụng lại Otsu trên ảnh đã được CLAHE "cứu"
ret_final, thresh_final = cv2.threshold(img_clahe_blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Hiển thị quá trình
show_images(
    [img_doc, img_clahe, thresh_final],
    ['1. Ảnh Gốc (Đổ bóng)', '2. Sau khi áp dụng CLAHE', f'3. Otsu Final (T={ret_final})'],
    cols=3
)

## PHẦN 4: LƯỢNG HÓA MÀU SẮC BẰNG PHÂN CỤM K-MEANS (NHIỆM VỤ LAB 3)

### 4.1. Mục tiêu và Ý tưởng thuật toán
Các kỹ thuật cắt ngưỡng ở trên chỉ tạo ra ảnh nhị phân (2 màu đen/trắng). Vậy nếu chúng ta muốn giữ lại màu sắc nhưng **giảm số lượng màu** xuống (ví dụ từ hàng triệu màu xuống còn 4, 8, hoặc 16 màu) thì sao? Đó gọi là quá trình **Lượng hóa màu sắc (Color Quantization)**.

**Thuật toán K-Means:**
* K-Means là một thuật toán **Học không giám sát (Unsupervised Learning)**.
* Nó sẽ gom nhóm các pixel có màu sắc tương đồng nhau vào cùng một "Cụm" (Cluster).
* Chúng ta cần chỉ định trước số lượng màu muốn giữ lại, ký hiệu là $K$.
* Thuật toán sẽ lặp đi lặp lại việc tìm ra $K$ màu đại diện (gọi là tâm cụm - centroids) và gán mỗi pixel về màu đại diện gần nó nhất.

**Ứng dụng:** Giảm dung lượng lưu trữ ảnh, tạo hiệu ứng nghệ thuật (Posterize), hoặc đơn giản hóa dữ liệu trước khi nhận dạng vật thể.

In [ ]:
# 1. Tạo một ảnh màu giả lập (Nếu bạn chưa tải ảnh lên)
# Gợi ý: Các bạn sinh viên nên tải lên một bức ảnh phong cảnh thật (ví dụ: đồi cà phê, bầu trời)
# để thấy rõ hiệu ứng nghệ thuật của thuật toán này.
img_h, img_w = 400, 600
dummy_landscape = np.zeros((img_h, img_w, 3), dtype=np.uint8)
# Tạo bầu trời xanh
dummy_landscape[0:200, :, :] = [255, 200, 100] # BGR cho màu xanh dương nhạt
# Tạo đồi cỏ xanh lá
dummy_landscape[200:400, :, :] = [100, 200, 50] # BGR cho màu xanh lá
# Thêm nhiễu màu ngẫu nhiên để mô phỏng chi tiết ảnh
noise = np.random.randint(-30, 30, (img_h, img_w, 3)).astype(np.int16)
dummy_landscape = np.clip(dummy_landscape + noise, 0, 255).astype(np.uint8)

cv2.imwrite('landscape_sample.jpg', dummy_landscape)

# ==========================================

# 2. Đọc ảnh gốc (Màu BGR)
land_path = 'landscape_sample.jpg'
img_color = cv2.imread(land_path)

# Hiển thị ảnh gốc để quan sát (nhớ dùng hàm show_images đã định nghĩa ở Phần 1)
show_images([img_color], ['Ảnh gốc (Hàng triệu màu sắc)'], cols=1, figsize=(6, 4))

### 4.2. Chuẩn bị dữ liệu cho OpenCV K-Means
Hàm `cv2.kmeans()` trong OpenCV yêu cầu dữ liệu đầu vào phải tuân thủ định dạng rất nghiêm ngặt:
1.  **Chuyển đổi vector:** Ảnh 3D `(Chiều cao, Chiều rộng, 3 kênh màu)` phải được "duỗi thẳng" thành ma trận 2D `(Số lượng Pixel, 3 kênh màu)`.
2.  **Kiểu dữ liệu:** Phải chuyển sang số thực `float32`.

### 4.3. Cấu hình tiêu chí dừng (Criteria)
K-Means là thuật toán lặp. Cần cho máy tính biết khi nào thì dừng lặp:
* `cv2.TERM_CRITERIA_EPS`: Dừng khi tâm cụm không dịch chuyển quá một khoảng Epsilon nhất định (đạt độ chính xác).
* `cv2.TERM_CRITERIA_MAX_ITER`: Dừng khi đạt số vòng lặp tối đa.
* Thường chúng ta sẽ kết hợp cả hai.

In [ ]:
# 1. Định dạng lại dữ liệu ảnh
# -1 nghĩa là tự động tính toán số lượng pixel dựa trên kích thước ban đầu
pixel_values = img_color.reshape((-1, 3))
pixel_values = np.float32(pixel_values)

print(f"Kích thước ảnh gốc: {img_color.shape}")
print(f"Kích thước mảng dữ liệu đưa vào K-Means: {pixel_values.shape}")

# 2. Định nghĩa tiêu chí dừng (Criteria)
# Dừng lặp khi: Đạt 100 vòng lặp HOẶC tâm cụm di chuyển ít hơn 0.2
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)

# 3. Hàm tiện ích để thực thi K-Means và tái tạo ảnh
def apply_kmeans(data, K, original_shape):
    # Áp dụng K-Means
    # attempts = 10: Chạy thuật toán 10 lần với các tâm cụm khởi tạo ngẫu nhiên khác nhau, lấy kết quả tốt nhất
    ret, labels, centers = cv2.kmeans(data, K, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)

    # Ép kiểu các tâm cụm (chính là K màu sắc mới) về số nguyên uint8 (0-255)
    centers = np.uint8(centers)

    # Tạo lại ảnh: gán mỗi pixel bằng màu của tâm cụm tương ứng
    # labels.flatten() chứa index cụm (từ 0 đến K-1) của từng pixel
    segmented_data = centers[labels.flatten()]

    # Định dạng lại thành ảnh 3D như ban đầu
    segmented_image = segmented_data.reshape(original_shape)
    return segmented_image

# 4. Thử nghiệm với các giá trị K khác nhau
img_k16 = apply_kmeans(pixel_values, 16, img_color.shape)
img_k8 = apply_kmeans(pixel_values, 8, img_color.shape)
img_k4 = apply_kmeans(pixel_values, 4, img_color.shape)

# 5. Hiển thị kết quả so sánh
show_images(
    [img_color, img_k16, img_k8, img_k4],
    ['Ảnh gốc', 'K=16 (16 Màu)', 'K=8 (8 Màu)', 'K=4 (4 Màu)'],
    rows=2, cols=2, figsize=(12, 8)
)

## PHẦN 5: PHÂN ĐOẠN DỰA TRÊN MẬT ĐỘ VỚI THUẬT TOÁN MEAN SHIFT

### 5.1. Khắc phục điểm yếu của K-Means
Trong Phần 4, K-Means hoạt động rất tốt nhưng đòi hỏi chúng ta phải "đoán" và gán cứng số lượng cụm $K$ (ví dụ $K=4$ hoặc $K=8$). Nếu chọn sai $K$, kết quả phân đoạn sẽ không tự nhiên.

**Thuật toán Mean Shift** ra đời để giải quyết vấn đề này:
* **Không cần biết trước $K$:** Thuật toán tự động tìm ra số lượng cụm phù hợp dựa trên sự phân bố dữ liệu thực tế.
* **Nguyên lý hoạt động (Tìm cực đại mật độ):** Máy tính sẽ tạo ra một "cửa sổ trượt" (sliding window) và liên tục dịch chuyển cửa sổ này về phía có mật độ điểm ảnh (cùng màu sắc) dày đặc nhất, cho đến khi chạm "đỉnh" (mode) và dừng lại.
* **Kết quả:** Các vùng màu có sự tương đồng sẽ được cào bằng (làm phẳng màu), tạo ra hiệu ứng giống như tranh vẽ (cartoon/painting effect), đồng thời vẫn giữ được độ sắc nét của đường biên (edge-preserving).

### 5.2. Lập trình Mean Shift với OpenCV
Trong OpenCV, chúng ta sử dụng hàm `cv2.pyrMeanShiftFiltering()`. Hàm này phân cụm dựa trên cả **miền không gian** (tọa độ x, y) và **miền màu sắc** (R, G, B).

* **Lưu ý quan trọng (Chi phí tính toán):** Mean Shift chạy chậm hơn K-Means rất nhiều! Vì vậy, trước khi chạy thuật toán, chúng ta BẮT BUỘC phải thu nhỏ kích thước ảnh (Resize) để tránh làm treo máy.

In [ ]:
# 1. Thu nhỏ kích thước ảnh để giảm chi phí tính toán cho Mean Shift
# Lấy lại ảnh phong cảnh từ Phần 4 (img_color)
# Thay đổi kích thước về chiều rộng 400px, chiều cao tỷ lệ tương ứng
h, w = img_color.shape[:2]
new_w = 400
new_h = int((new_w / w) * h)
img_small = cv2.resize(img_color, (new_w, new_h))

# 2. Áp dụng thuật toán Mean Shift
# Các tham số quan trọng:
# - sp (Spatial Window Radius): Bán kính không gian (càng lớn, các vùng ở xa nhau càng dễ gộp lại).
# - sr (Color Window Radius): Bán kính màu sắc (càng lớn, các màu hơi khác nhau cũng bị gộp chung).
print("Đang chạy thuật toán Mean Shift... (Có thể mất vài giây)")
img_meanshift = cv2.pyrMeanShiftFiltering(img_small, sp=20, sr=40)
print("✅ Hoàn thành!")

# 3. Áp dụng thử K-Means trên cùng bức ảnh nhỏ này (để so sánh công bằng)
# Thử với K=8
data_small = np.float32(img_small.reshape((-1, 3)))
ret, labels, centers = cv2.kmeans(data_small, 8, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
img_kmeans_small = np.uint8(centers)[labels.flatten()].reshape(img_small.shape)

# 4. Hiển thị so sánh 3 phương pháp
show_images(
    [img_small, img_kmeans_small, img_meanshift],
    ['1. Ảnh gốc (Đã thu nhỏ)', '2. K-Means (K=8, Nhanh)', '3. Mean Shift (Chậm, Tự nhiên)'],
    cols=3, figsize=(15, 5)
)

### 5.3. Thảo luận và Đánh giá (CLO4)
Quan sát kết quả ở trên, các bạn có thể rút ra sự khác biệt cốt lõi giữa 2 phương pháp phân cụm:
1. **K-Means:** Chạy rất nhanh, phù hợp để tối ưu dung lượng ảnh hoặc tạo hiệu ứng giảm màu diện rộng. Tuy nhiên, nó không quan tâm đến vị trí tọa độ của pixel, nên các điểm màu giống nhau dù ở xa nhau vẫn bị gộp chung.
2. **Mean Shift:** Chạy chậm và tốn tài nguyên, nhưng kết quả phân đoạn rất tự nhiên, giữ được sự liền mạch của các mảng màu và không làm nhòe ranh giới giữa các vật thể.

## PHẦN 6: TÁCH NỀN VẬT THỂ BẰNG LÁT CẮT ĐỒ THỊ (GRABCUT - NHIỆM VỤ LAB 3)

### 6.1. Từ Đồ thị đến thuật toán GrabCut
Các phương pháp cắt ngưỡng hay K-Means đều có nhược điểm là dễ làm đối tượng bị "vụn" nếu màu sắc nền quá phức tạp. Thuật toán **GrabCut** giải quyết triệt để bài toán này bằng cách chuyển đổi bức ảnh thành một đồ thị $G = (V, E)$:
* **Đỉnh (V):** Mỗi pixel là một đỉnh. Có 2 đỉnh đặc biệt là Nguồn $S$ (đại diện cho vật thể) và Cống $T$ (đại diện cho nền).
* **Cạnh (E):** Các pixel lân cận có màu giống nhau sẽ có liên kết rất chặt (khó cắt).
* **Lát cắt tối thiểu (Min-cut):** Thuật toán sẽ tìm ra một nhát cắt đi qua các vùng có sự chênh lệch màu sắc lớn nhất, qua đó tách rời $S$ và $T$.

**Cách GrabCut hoạt động (Tương tác):**
Bạn chỉ cần cung cấp một **khung chữ nhật (Bounding Box)** bao quanh vật thể. Thuật toán sẽ mặc định mọi thứ bên ngoài khung là Nền, bên trong khung là Vật thể (chứa cả nền nhiễu). Sau đó, nó dùng mô hình Gaussian (GMM) để tự động học và bóc tách phần nền còn sót lại bên trong khung.

In [ ]:
# 1. Tạo một bức ảnh giả lập có bông hoa ở giữa (Nền phức tạp)
# Khuyến khích: Sinh viên tải lên ảnh thực tế (ví dụ: một bông hoa cà phê trên nền lá)
img_h, img_w = 400, 600
dummy_flower = np.zeros((img_h, img_w, 3), dtype=np.uint8)
# Nền lá cây lộn xộn
dummy_flower[:, :] = [50, 150, 50]
noise = np.random.randint(-40, 40, (img_h, img_w, 3)).astype(np.int16)
dummy_flower = np.clip(dummy_flower + noise, 0, 255).astype(np.uint8)
# Bông hoa màu đỏ ở giữa
cv2.circle(dummy_flower, (300, 200), 80, (0, 0, 255), -1)
cv2.circle(dummy_flower, (300, 200), 30, (0, 255, 255), -1) # Nhụy vàng
cv2.imwrite('flower_sample.jpg', dummy_flower)

# ==========================================

# 2. Đọc ảnh cần tách nền
flower_path = 'flower_sample.jpg'
img_target = cv2.imread(flower_path)

# 3. Khởi tạo các biến cần thiết cho GrabCut
# Tạo mặt nạ ban đầu toàn số 0
mask = np.zeros(img_target.shape[:2], np.uint8)

# Mô hình nền và vật thể dùng nội bộ cho thuật toán (bắt buộc phải là mảng 1x65 float64)
bgdModel = np.zeros((1, 65), np.float64)
fgdModel = np.zeros((1, 65), np.float64)

# 4. Xác định khung chữ nhật chứa vật thể (x, y, width, height)
# Bạn cần điều chỉnh tọa độ này sao cho nó bao quanh vừa vặn vật thể trong ảnh thật của bạn
rect = (200, 100, 200, 200)

# 5. Thực thi GrabCut
print("Đang chạy thuật toán GrabCut (5 vòng lặp)...")
# iterCount = 5: Số vòng lặp tinh chỉnh để kết quả chính xác hơn
cv2.grabCut(img_target, mask, rect, bgdModel, fgdModel, iterCount=5, mode=cv2.GC_INIT_WITH_RECT)
print("✅ Hoàn thành!")

# 6. Xử lý mặt nạ đầu ra
# GrabCut trả về mask với 4 giá trị: 0 (Nền chắc chắn), 1 (Vật thể chắc chắn), 2 (Có thể là nền), 3 (Có thể là vật thể)
# Ta sẽ gộp 0 và 2 thành 0 (Nền), 1 và 3 thành 1 (Vật thể)
mask_binary = np.where((mask == 2) | (mask == 0), 0, 1).astype('uint8')

# Nhân ảnh gốc với mặt nạ nhị phân để lấy ra vật thể
img_extracted = img_target * mask_binary[:, :, np.newaxis]

# 7. Hiển thị kết quả
# Vẽ khung chữ nhật lên ảnh gốc để minh họa
img_rect = img_target.copy()
cv2.rectangle(img_rect, (rect[0], rect[1]), (rect[0]+rect[2], rect[1]+rect[3]), (255, 0, 0), 3)

show_images(
    [img_rect, mask_binary * 255, img_extracted],
    ['1. Ảnh gốc & Khung bao (Rect)', '2. Mặt nạ GrabCut (Mask)', '3. Vật thể được bóc tách'],
    cols=3, figsize=(15, 5)
)

### 6.2. Hậu xử lý (Post-processing) hoàn thiện Pipeline
Theo Rubric chấm điểm của Bài Lab 3, để đạt điểm Tối ưu (30%), chúng ta cần hoàn thiện quy trình xử lý.

Đôi khi kết quả GrabCut thô sẽ để lại những "lỗ hổng" nhỏ bên trong vật thể hoặc những "hạt nhiễu" li ti ngoài nền. Chúng ta sẽ áp dụng các **phép toán Hình thái học (Morphology)** để dọn dẹp mặt nạ này:
* **Phép Mở (Opening):** Xóa các đốm nhiễu nhỏ ngoài nền.
* **Phép Đóng (Closing):** Lấp các lỗ hổng nhỏ bên trong vật thể.

In [ ]:
# 1. Tạo Kernel (Ma trận cấu trúc) kích thước 5x5
kernel = np.ones((5, 5), np.uint8)

# 2. Làm sạch mặt nạ bằng Hình thái học
# Áp dụng Closing để lấp lỗ hổng trong vật thể, sau đó Opening để xóa nhiễu nền
mask_cleaned = cv2.morphologyEx(mask_binary, cv2.MORPH_CLOSE, kernel)
mask_cleaned = cv2.morphologyEx(mask_cleaned, cv2.MORPH_OPEN, kernel)

# 3. Áp dụng mặt nạ đã làm sạch lên ảnh gốc
img_final = img_target * mask_cleaned[:, :, np.newaxis]

# 4. Hiển thị so sánh trước và sau khi Hậu xử lý
show_images(
    [mask_binary * 255, mask_cleaned * 255, img_final],
    ['Mặt nạ thô (GrabCut)', 'Mặt nạ đã làm sạch (Morphology)', 'Kết quả Tách nền Cuối cùng'],
    cols=3, figsize=(15, 5)
)

## PHẦN NÂNG CAO 1

## PHẦN 7: PHÂN ĐOẠN VẬT THỂ DÍNH LIỀN VỚI THUẬT TOÁN WATERSHED (ĐƯỜNG PHÂN THỦY)

### 7.1. Vấn đề của các thuật toán cơ bản
Hãy tưởng tượng bạn đang viết phần mềm đếm số lượng hạt cà phê trong một rổ, hoặc đếm số lượng tế bào dưới kính hiển vi. Khi các vật thể này nằm **dính sát vào nhau**, Cắt ngưỡng (Otsu) hay K-Means sẽ coi toàn bộ cụm dính liền đó là *một vật thể duy nhất*.

Để giải quyết, chúng ta sử dụng **Thuật toán Watershed (Đường phân thủy)**.

### 7.2. Nguyên lý hoạt động của Watershed
Thuật toán này coi bức ảnh xám như một **bản đồ địa hình** (Topographic surface):
* Những vùng sáng màu (vật thể) được coi là các **đỉnh núi** (Peaks).
* Những vùng tối màu (nền) được coi là các **thung lũng** (Valleys).
* Thuật toán sẽ mô phỏng việc "bơm nước" từ các thung lũng lên. Khi nước dâng lên và chuẩn bị tràn từ thung lũng này sang thung lũng khác (nơi 2 vật thể chạm nhau), máy tính sẽ xây một "con đập" để ngăn lại. Các con đập này chính là đường ranh giới phân tách các vật thể!

### 7.3. Quy trình thực hiện (Pipeline)
Để máy tính bơm nước đúng chỗ, ta không để nó tự làm mò mà phải đánh dấu (Marker-based Watershed):
1. **Tìm vùng chắc chắn là Nền (Sure Background):** Bằng cách làm phình (Dilation) vật thể ra.
2. **Tìm vùng chắc chắn là Vật thể (Sure Foreground):** Sử dụng phép biến đổi khoảng cách **Distance Transform** để tìm ra "lõi" hay "tâm" của từng vật thể, bỏ qua phần rìa đang dính nhau.
3. **Tìm vùng Không xác định (Unknown):** Lấy Nền trừ đi Vật thể. Đây là khu vực ranh giới đang xảy ra tranh chấp.
4. **Đánh dấu (Marker) và chạy Watershed:** Nước sẽ dâng từ các lõi (Sure FG) và dừng lại ở vùng Unknown để tạo ranh giới.

In [ ]:
# 1. Tạo ảnh giả lập các hạt/đồng xu dính liền nhau
# Khuyến khích: Sinh viên tải lên ảnh một cụm hạt cà phê dính nhau trên nền sáng
img_h, img_w = 400, 400
dummy_coins = np.ones((img_h, img_w, 3), dtype=np.uint8) * 255 # Nền trắng
# Vẽ 3 hình tròn (giả lập vật thể) dính sát vào nhau
cv2.circle(dummy_coins, (150, 150), 60, (50, 50, 50), -1)
cv2.circle(dummy_coins, (240, 150), 60, (50, 50, 50), -1)
cv2.circle(dummy_coins, (195, 230), 60, (50, 50, 50), -1)
# Thêm chút nhiễu
noise = np.random.randint(-20, 20, (img_h, img_w, 3)).astype(np.int16)
dummy_coins = np.clip(dummy_coins + noise, 0, 255).astype(np.uint8)
cv2.imwrite('coins_sample.jpg', dummy_coins)

# ==========================================

# 2. Tiền xử lý và Cắt ngưỡng Otsu
img_coins = cv2.imread('coins_sample.jpg')
gray = cv2.cvtColor(img_coins, cv2.COLOR_BGR2GRAY)
# Vì vật thể tối trên nền sáng, ta dùng THRESH_BINARY_INV để đảo ngược (Vật thể thành trắng, nền thành đen)
ret, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

# 3. Loại bỏ nhiễu bằng Hình thái học (Opening)
kernel = np.ones((3,3), np.uint8)
opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)

# 4. Tìm vùng CHẮC CHẮN LÀ NỀN (Sure Background) bằng cách Dilation (Làm phình vật thể)
sure_bg = cv2.dilate(opening, kernel, iterations=3)

# 5. Tìm vùng CHẮC CHẮN LÀ VẬT THỂ (Sure Foreground) bằng Distance Transform
# Distance Transform tính khoảng cách từ mỗi pixel trắng đến pixel đen gần nhất.
# Những pixel nằm sâu bên trong "lõi" sẽ có giá trị cao nhất.
dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
# Lấy ngưỡng bằng 50% giá trị cực đại để thu được các "tâm" tách rời nhau
ret, sure_fg = cv2.threshold(dist_transform, 0.5 * dist_transform.max(), 255, 0)
sure_fg = np.uint8(sure_fg)

# 6. Tìm vùng KHÔNG XÁC ĐỊNH (Unknown) - Nơi ranh giới dính liền
unknown = cv2.subtract(sure_bg, sure_fg)

# Hiển thị các bước trung gian để dễ hình dung
show_images(
    [thresh, sure_bg, dist_transform, sure_fg],
    ['1. Mask Ban đầu (Dính nhau)', '2. Chắc chắn là Nền', '3. Distance Transform', '4. Chắc chắn là Vật thể (Lõi)'],
    rows=2, cols=2, figsize=(10, 8)
)

### 7.4. Đánh dấu và Thực thi Watershed
Sau khi đã có "Lõi" của vật thể và "Nền", chúng ta sẽ dán nhãn (label) cho chúng. Hãy tưởng tượng như việc ta đánh số 1, 2, 3 cho từng hạt cà phê, đánh số 0 cho phần Không xác định.

Sau đó, hàm `cv2.watershed()` sẽ dựa vào bản đồ này để bắt đầu phân đoạn. Nơi nào được xác định là đường ranh giới phân cắt, thuật toán sẽ gán cho nó giá trị `-1`.

In [ ]:
# 1. Đánh dấu các vùng độc lập (Connected Components)
# Hàm này sẽ đánh số ID cho các vùng trắng tách biệt trong sure_fg (ví dụ: hạt thứ 1 là ID 1, hạt thứ 2 là ID 2)
# Nền đen sẽ mang ID 0.
ret, markers = cv2.connectedComponents(sure_fg)

# 2. Điều chỉnh Marker
# Cộng thêm 1 vào tất cả các marker để Nền chắc chắn mang ID 1 (thay vì 0)
markers = markers + 1

# Gán ID 0 cho những vùng KHÔNG XÁC ĐỊNH (vùng tranh chấp ranh giới) để Watershed biết đường mà chạy
markers[unknown == 255] = 0

# 3. Thực thi thuật toán Watershed
# Thuật toán sẽ sửa đổi trực tiếp mảng markers. Ranh giới tìm được sẽ có giá trị -1.
img_watershed_result = img_coins.copy()
markers = cv2.watershed(img_watershed_result, markers)

# 4. Vẽ ranh giới phân thủy lên ảnh gốc
# Tô màu đỏ (0, 0, 255) cho các pixel mang giá trị -1 (đường ranh giới)
img_watershed_result[markers == -1] = [0, 0, 255]

# Hiển thị kết quả cuối cùng
# Trực quan hóa mảng markers bằng color map để thấy rõ từng vật thể đã được phân tách
marker_vis = np.uint8(np.interp(markers, [markers.min(), markers.max()], [0, 255]))
marker_vis = cv2.applyColorMap(marker_vis, cv2.COLORMAP_JET)

show_images(
    [img_coins, marker_vis, img_watershed_result],
    ['Ảnh gốc (Các vật thể dính nhau)', 'Bản đồ Marker (Sau Watershed)', 'Kết quả phân tách ranh giới'],
    cols=3, figsize=(15, 5)
)

## PHẦN NÂNG CAO 2

## PHẦN 8: TÁCH NỀN THỜI GIAN THỰC TRÊN VIDEO (BACKGROUND SUBTRACTION)

### 8.1. Động lực (Motivation)
Từ Phần 1 đến Phần 7, chúng ta chỉ mới xử lý các bức ảnh tĩnh. Tuy nhiên, trong các bài toán Thị giác máy tính ứng dụng thực tế (như camera an ninh, đếm lưu lượng xe cộ, xe tự lái), dữ liệu đầu vào là một **luồng video liên tục**.

Làm thế nào để máy tính biết được đâu là vật thể đang chuyển động (con người, xe cộ) và đâu là phông nền tĩnh (đường phố, tòa nhà)? Chúng ta không thể dùng GrabCut cho từng khung hình vì nó quá chậm và yêu cầu tương tác thủ công. Giải pháp ở đây là kỹ thuật **Background Subtraction (Trừ Nền)**.

### 8.2. Thuật toán MOG2 (Mixture of Gaussians)
Thay vì dùng một bức ảnh nền tĩnh cố định để trừ đi, OpenCV cung cấp thuật toán `cv2.createBackgroundSubtractorMOG2()`.
* **Cơ chế học:** Nó quan sát một số lượng khung hình nhất định (history) để tự động "học" và cập nhật xem cái gì là nền. Nhờ đó, nó có thể thích nghi với sự thay đổi của ánh sáng trong ngày (từ sáng sang chiều).
* **Phát hiện bóng râm (Shadow Detection):** Tính năng vượt trội của MOG2 là khả năng nhận diện bóng của vật thể in xuống mặt đường và đánh dấu bóng râm bằng màu xám (127), giúp phân biệt rõ ràng với bản thể thực sự của vật thể (màu trắng - 255).

### 8.3. Thực hành trên Jupyter Notebook
Trong môi trường Jupyter/Colab, việc hiển thị video trực tiếp bằng `cv2.imshow()` thường gây treo trình duyệt. Do đó, chúng ta sẽ giả lập một đoạn video ngắn (gồm nhiều khung hình liên tiếp) và trích xuất một vài khung hình tiêu biểu để quan sát cách thuật toán bóc tách vật thể chuyển động.

In [ ]:
# 1. Giả lập một đoạn video ngắn (Danh sách các khung hình/frames)
# Tạo một phông nền tĩnh (Ví dụ: một đoạn đường có bốt bảo vệ màu xám đậm)
background = np.ones((300, 500, 3), dtype=np.uint8) * 150
cv2.rectangle(background, (50, 50), (150, 250), (100, 100, 100), -1)

frames = []
# Khởi tạo 20 khung hình với một vật thể (chiếc xe hình tròn màu đỏ) di chuyển ngang qua
for i in range(20):
    frame = background.copy()

    # Vị trí x thay đổi theo thời gian để tạo hiệu ứng di chuyển
    x_pos = 50 + i * 20
    cv2.circle(frame, (x_pos, 150), 35, (0, 0, 255), -1)

    # Thêm nhiễu ngẫu nhiên vào mỗi khung hình để mô phỏng noise của camera thực tế
    noise = np.random.randint(-15, 15, frame.shape).astype(np.int16)
    frame = np.clip(frame + noise, 0, 255).astype(np.uint8)

    frames.append(frame)

# ==========================================

# 2. Khởi tạo thuật toán Trừ nền MOG2
# history: Số lượng frame dùng để mô hình hóa nền.
# varThreshold: Ngưỡng quyết định pixel có thuộc vật thể chuyển động hay không (càng nhỏ càng nhạy).
# detectShadows=True: Bật tính năng nhận diện bóng râm.
fgbg = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=16, detectShadows=True)

# 3. Chạy vòng lặp xử lý từng khung hình trong video
masks = []
for frame in frames:
    # Hàm apply() liên tục học nền và trả về mặt nạ nhị phân của vật thể chuyển động
    fgmask = fgbg.apply(frame)
    masks.append(fgmask)

# 4. Hiển thị kết quả so sánh ở đầu và cuối đoạn video
# Sử dụng hàm show_images đã định nghĩa ở Phần 1
show_images(
    [frames[2], masks[2], frames[15], masks[15]],
    ['Khung hình số 2', 'Mặt nạ Tách nền (Frame 2)', 'Khung hình số 15', 'Mặt nạ Tách nền (Frame 15)'],
    rows=2, cols=2, figsize=(12, 8)
)

### 8.4. Hướng dẫn chạy Video thực tế trên máy tính cá nhân
Mã nguồn ở trên được thiết kế riêng để hiển thị an toàn trên Jupyter Notebook. Nếu các bạn sinh viên muốn làm dự án thực tế trên máy tính cá nhân (chạy bằng VS Code hoặc Pycharm) và kết nối trực tiếp với Webcam, hãy sử dụng cấu trúc vòng lặp `while` tiêu chuẩn như sau:

In [ ]:
import cv2

# Mở Webcam (số 0) hoặc đường dẫn file video ('video.mp4')
cap = cv2.VideoCapture(0)
fgbg = cv2.createBackgroundSubtractorMOG2(detectShadows=True)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Áp dụng thuật toán trừ nền
    fgmask = fgbg.apply(frame)

    # Hiển thị cửa sổ video
    cv2.imshow('Camera Gốc', frame)
    cv2.imshow('Mặt nạ Chuyển động', fgmask)

    # Bấm phím 'q' hoặc 'ESC' để thoát
    key = cv2.waitKey(30) & 0xff
    if key == 27 or key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

## PHẦN NÂNG CAO 3

## PHẦN 9: TRIỂN KHAI ỨNG DỤNG PHÂN ĐOẠN TƯƠNG TÁC VỚI STREAMLIT

### 9.1. Động lực (Motivation)
Từ đầu đến giờ, chúng ta phải sửa trực tiếp từng dòng code (như đổi $T=127$ thành $T=200$, hay $K=4$ thành $K=8$) rồi chạy lại cell để xem kết quả. Điều này không thân thiện với người dùng phổ thông.

**Giải pháp:** Chúng ta sẽ sử dụng thư viện **Streamlit** - một công cụ mã nguồn mở giúp lập trình viên Python chuyển đổi các đoạn code Data Science/Machine Learning thành một ứng dụng Web tương tác chỉ trong vài phút, không cần biết HTML/CSS hay JavaScript!

### 9.2. Cách Streamlit hoạt động trong môi trường Notebook
Streamlit được thiết kế để chạy từ Terminal (Dòng lệnh). Để chạy trên Jupyter Notebook hoặc Google Colab, chúng ta sẽ làm theo 2 bước:
1. Sử dụng lệnh ma thuật `%%writefile app.py` để ghi đoạn code Python dưới đây thành một file thực thi.
2. Chạy file đó bằng lệnh `!streamlit run app.py`.

In [ ]:
# Cài đặt thư viện streamlit nếu máy bạn chưa có
# !pip install streamlit

# LỆNH GHI FILE: Toàn bộ code bên dưới dòng này sẽ được lưu thành file app.py
%%writefile app.py
import streamlit as st
import cv2
import numpy as np
from PIL import Image

# 1. Tiêu đề và Cấu hình trang Web
st.set_page_config(page_title="App Phân Đoạn Ảnh", layout="wide")
st.title("📸 Ứng Dụng Phân Đoạn Ảnh Tương Tác")
st.markdown("**Môn học:** Thị giác máy tính | **Giảng viên:** Trần Thành Thắng")

# 2. Tạo thanh bên (Sidebar) để người dùng điều khiển
st.sidebar.header("⚙️ Bảng Điều Khiển")

# Nút Upload ảnh
uploaded_file = st.sidebar.file_uploader("Tải ảnh của bạn lên (JPG, PNG)", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    # Đọc ảnh từ file upload (chuyển sang định dạng OpenCV xử lý được)
    image = Image.open(uploaded_file)
    img_array = np.array(image)

    # OpenCV mặc định dùng BGR, ta cần chuyển từ RGB (của PIL) sang BGR để xử lý
    img_bgr = cv2.cvtColor(img_array, cv2.COLOR_RGB2BGR)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # Lựa chọn Thuật toán
    method = st.sidebar.selectbox("Chọn Thuật Toán Phân Đoạn:",
                                  ("Cắt Ngưỡng Toàn Cục", "Lượng Hóa Màu K-Means"))

    st.markdown("---")
    col1, col2 = st.columns(2)

    # Hiển thị ảnh gốc ở cột 1
    with col1:
        st.subheader("Ảnh Gốc")
        st.image(image, use_container_width=True)

    # Xử lý theo thuật toán được chọn và hiển thị ở cột 2
    with col2:
        st.subheader("Ảnh Kết Quả")

        if method == "Cắt Ngưỡng Toàn Cục":
            # Tạo thanh trượt (Slider) để người dùng chọn ngưỡng T trực tiếp trên web
            T_val = st.sidebar.slider("Chọn Ngưỡng (T):", min_value=0, max_value=255, value=127, step=1)

            # Tiền xử lý và Cắt ngưỡng
            blurred = cv2.GaussianBlur(img_gray, (5, 5), 0)
            ret, thresh_img = cv2.threshold(blurred, T_val, 255, cv2.THRESH_BINARY)

            st.image(thresh_img, use_container_width=True, channels="GRAY")
            st.success(f"Đã phân đoạn với Ngưỡng T = {T_val}")

        elif method == "Lượng Hóa Màu K-Means":
            # Tạo thanh trượt chọn số lượng màu K
            K_val = st.sidebar.slider("Chọn số màu (K):", min_value=2, max_value=32, value=8, step=1)

            if st.sidebar.button("Bắt đầu xử lý (Sẽ mất vài giây)"):
                with st.spinner("Đang chạy thuật toán K-Means..."):
                    # Định dạng dữ liệu cho K-Means
                    pixel_values = np.float32(img_bgr.reshape((-1, 3)))
                    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)

                    # Chạy K-Means
                    ret, labels, centers = cv2.kmeans(pixel_values, K_val, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)

                    # Tái tạo ảnh
                    centers = np.uint8(centers)
                    segmented_data = centers[labels.flatten()]
                    img_kmeans = segmented_data.reshape(img_bgr.shape)

                    # Chuyển lại sang RGB để Streamlit hiển thị đúng màu
                    img_kmeans_rgb = cv2.cvtColor(img_kmeans, cv2.COLOR_BGR2RGB)

                    st.image(img_kmeans_rgb, use_container_width=True)
                    st.success(f"Đã lượng hóa ảnh xuống còn {K_val} màu!")
else:
    st.info("👈 Vui lòng tải một bức ảnh lên từ thanh bên trái để bắt đầu!")

### 9.3. Hướng dẫn khởi chạy Web App
**Nếu bạn đang chạy trên máy tính cá nhân (Local):**
1. Mở Terminal (Command Prompt / PowerShell).
2. Di chuyển đến thư mục chứa file `app.py` vừa tạo.
3. Gõ lệnh: `streamlit run app.py`
4. Một trang web sẽ tự động mở lên trên trình duyệt của bạn (thường ở địa chỉ `localhost:8501`).

**Nếu bạn đang chạy trên Google Colab:**
Việc chạy Streamlit trên Colab cần dùng một mẹo nhỏ để mở cổng mạng ảo (Tunnel). Hãy chạy cell bên dưới:

In [ ]:
# Cell này dành riêng cho Google Colab để lấy link public web app
import urllib
print("Vui lòng click vào link có đuôi '.loca.lt' bên dưới.")
print("Mật khẩu (Endpoint IP) để nhập vào trang web là:")
!wget -q -O - ipv4.icanhazip.com

# Khởi chạy streamlit ngầm và xuất ra public url bằng localtunnel
!npm install localtunnel
!streamlit run app.py & npx localtunnel --port 8501

---
### 🌟 TỔNG KẾT BÀI THỰC HÀNH LAB 3
Chúc mừng các bạn đã hoàn thành xuất sắc Chương 3: Phân đoạn ảnh!
Chúng ta đã đi một chặng đường dài từ việc:
1.  **Cắt ngưỡng đơn giản** và tự động hóa với **Otsu**.
2.  Xử lý ánh sáng phức tạp bằng **CLAHE + Adaptive Thresholding**.
3.  Tạo hiệu ứng giảm màu nghệ thuật bằng **K-Means**.
4.  Hoàn thiện hệ thống tách nền vật thể chuyên nghiệp bằng **GrabCut & Hình thái học**.

Hãy lưu lại file Jupyter Notebook này, hoàn thiện báo cáo và nộp lên hệ thống LMS theo đúng hạn nhé!

# BÀI TẬP TỰ LUYỆN CHƯƠNG 3: PHÂN ĐOẠN ẢNH

### Bài 1: Nhị phân hóa ảnh cơ bản (Global Thresholding)
* **Mục tiêu:** Hiểu cách hoạt động của ngưỡng toàn cục.
* **Mô tả:** Tải một bức ảnh có độ tương phản cao (ví dụ: một logo đen trên nền trắng hoặc bóng râm rõ nét). Trích xuất vật thể ra khỏi nền.
* **Hướng dẫn:** 1. Chuyển ảnh sang thang độ xám (`cv2.cvtColor`).
    2. Vẽ biểu đồ Histogram (`plt.hist`) để quan sát hai đỉnh phân bố.
    3. Chọn một giá trị $T$ nằm ở "thung lũng" giữa hai đỉnh và dùng `cv2.threshold` với cờ `cv2.THRESH_BINARY`.

### Bài 2: Tự động hóa với thuật toán Otsu
* **Mục tiêu:** Sử dụng máy tính để tự tìm ngưỡng tối ưu.
* **Mô tả:** Sử dụng lại bức ảnh ở Bài 1, nhưng lần này không truyền giá trị $T$ thủ công nữa.
* **Hướng dẫn:** 1. Dùng `cv2.threshold` kết hợp cờ `cv2.THRESH_BINARY + cv2.THRESH_OTSU`.
    2. In ra giá trị ngưỡng $T$ mà thuật toán Otsu vừa tìm được và so sánh với giá trị bạn tự chọn ở Bài 1 xem có sát nhau không.

### Bài 3: "Cứu" ảnh tài liệu lóa sáng (Adaptive Thresholding)
* **Mục tiêu:** Xử lý ảnh có điều kiện chiếu sáng không đồng đều.
* **Mô tả:** Chụp một trang sách hoặc tài liệu học tập dưới ánh đèn bàn sao cho một góc bị chói sáng, góc kia bị đổ bóng. Yêu cầu làm rõ nét chữ trên toàn bộ trang.
* **Hướng dẫn:** 1. Tiền xử lý bằng Gaussian Blur.
    2. Sử dụng `cv2.adaptiveThreshold` với phương pháp `cv2.ADAPTIVE_THRESH_GAUSSIAN_C`.
    3. Thử tinh chỉnh tham số `blockSize` (ví dụ: 11, 21) và hằng số `C` (ví dụ: 2, 5, 10) để xem nét chữ thay đổi độ dày mỏng ra sao.

### Bài 4: Lượng hóa màu sắc phong cảnh (K-Means)
* **Mục tiêu:** Giảm dung lượng và tạo hiệu ứng nghệ thuật.
* **Mô tả:** Tải một bức ảnh phong cảnh địa phương (ví dụ: đồi cà phê, thác nước) có độ phân giải vừa phải. Giảm số lượng màu của ảnh xuống chỉ còn 6 màu.
* **Hướng dẫn:** 1. Chuyển đổi dữ liệu ảnh thành mảng 2D kiểu `float32`.
    2. Thiết lập tiêu chí dừng `criteria`.
    3. Chạy `cv2.kmeans` với $K=6$.
    4. Ép kiểu kết quả về `uint8` và tái tạo lại hình dạng ảnh 3D ban đầu.

### Bài 5: So sánh K-Means và Mean Shift
* **Mục tiêu:** Đánh giá ưu/nhược điểm của hai thuật toán phân cụm.
* **Mô tả:** Chạy thuật toán K-Means (với $K$ bất kỳ) và thuật toán Mean Shift trên cùng một bức ảnh (đã được thu nhỏ kích thước).
* **Hướng dẫn:** 1. Dùng `cv2.resize` để giảm kích thước ảnh gốc xuống còn khoảng 300x300 pixel.
    2. Chạy `cv2.pyrMeanShiftFiltering(img, sp=20, sr=40)`.
    3. Dùng module `time` trong Python để đo và in ra thời gian chạy của từng thuật toán. Rút ra nhận xét.

### Bài 6: Tách nền vật thể trung tâm (GrabCut cơ bản)
* **Mục tiêu:** Tách một vật thể ra khỏi nền phức tạp bằng khung tương tác.
* **Mô tả:** Tải ảnh một đồ vật nằm giữa khung hình (ví dụ: một ly cà phê trên bàn).
* **Hướng dẫn:** 1. Khởi tạo `mask`, `bgdModel`, `fgdModel`.
    2. Xác định tọa độ khung chữ nhật `rect` bao quanh ly cà phê.
    3. Chạy `cv2.grabCut` với chế độ `cv2.GC_INIT_WITH_RECT`.
    4. Lọc mask để giữ lại nhãn 1 và 3 (vật thể), sau đó nhân với ảnh gốc để xuất kết quả.

### Bài 7: Làm sạch mặt nạ với Hình thái học
* **Mục tiêu:** Hậu xử lý kết quả phân đoạn.
* **Mô tả:** Lấy kết quả mặt nạ nhị phân (mask) từ Bài 6. Rất có thể mặt nạ này còn những lỗ hổng nhỏ bên trong hoặc đốm nhiễu bên ngoài.
* **Hướng dẫn:** 1. Tạo một kernel kích thước 5x5 bằng `np.ones`.
    2. Dùng `cv2.morphologyEx` để thực hiện phép Đóng (Closing) nhằm lấp lỗ hổng.
    3. Tiếp tục dùng phép Mở (Opening) để xóa đốm nhiễu nền.
    4. Hiển thị ảnh vật thể cuối cùng.

### Bài 8: Đếm số lượng vật thể dính liền (Watershed)
* **Mục tiêu:** Phân tách các đối tượng nằm sát nhau.
* **Mô tả:** Tìm một bức ảnh chụp nhiều vật thể hình tròn dính sát nhau (ví dụ: một vốc hạt cà phê, hoặc nhiều đồng xu để sát nhau). Yêu cầu tách rời chúng ra.
* **Hướng dẫn:** 1. Nhị phân hóa ảnh bằng Otsu.
    2. Dùng `cv2.distanceTransform` để tìm phần "lõi" của từng hạt.
    3. Cắt ngưỡng phần lõi này, sau đó dùng `cv2.connectedComponents` để đánh dấu (marker).
    4. Chạy `cv2.watershed` để tìm ranh giới phân cắt.

### Bài 9: Nhận diện chuyển động cơ bản (Background Subtraction)
* **Mục tiêu:** Làm quen với dữ liệu thời gian thực.
* **Mô tả:** Tải một đoạn video ngắn (khoảng 5-10 giây) quay cảnh đường phố có xe cộ qua lại. Trích xuất các phương tiện đang di chuyển ra khỏi mặt đường.
* **Hướng dẫn:** 1. Khởi tạo `cv2.VideoCapture('video.mp4')`.
    2. Khởi tạo bộ trừ nền `cv2.createBackgroundSubtractorMOG2(detectShadows=True)`.
    3. Dùng vòng lặp `while` để đọc từng frame và truyền vào hàm `apply()`. Hiển thị mặt nạ nhị phân (mask) thu được.

### Bài 10: Xây dựng Giao diện Tương tác Mini (Tích hợp Streamlit)
* **Mục tiêu:** Đưa code thuật toán lên nền tảng Web.
* **Mô tả:** Viết một đoạn mã để tạo một trang web nhỏ, cho phép người dùng upload ảnh lên và dùng thanh trượt để chỉnh ngưỡng $T$ (như đã học ở phần cuối).
* **Hướng dẫn:** 1. Dùng magic command `%%writefile app.py`.
    2. Gọi `st.file_uploader` để nhận ảnh, `st.slider` để nhận giá trị $T$.
    3. Áp dụng `cv2.threshold` và hiển thị kết quả bằng `st.image`.
    4. (Tuỳ chọn) Bạn có thể suy nghĩ thêm cách tích hợp ứng dụng này với một bài trắc nghiệm nhanh (quiz) trên cùng giao diện Streamlit để kiểm tra kiến thức lý thuyết!